# 111 — Load Layer 1 Entities into Neo4j

Ingests the ICIJ Offshore Leaks subgraph prepared upstream (`data/layer_1/entities/` + `data/layer_1/links/`) into Neo4j.

Creates five node labels (`Person`, `Company`, `Intermediary`, `Address`, `Jurisdiction`) and six relationship types (`IS_OFFICER_OF`, `INTERMEDIARY_OF`, `REGISTERED_AT`, `SHARES_ADDRESS_WITH`, `SIMILAR_TO`, `INCORPORATED_IN`).

Account and Transaction nodes are intentionally out of scope for this build.

**Prerequisites**
- `.env` with `NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`
- CSVs already placed in `data/layer_1/entities/` and `data/layer_1/links/`

**Output**: Neo4j populated with the full Layer 1 graph. Summary counts printed at the end.

## Imports and connection

In [1]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))
load_dotenv(REPO / '.env')

from src.graph.connection import Neo4jConnection

DATA = REPO / 'data' / 'layer_1'
ENT = DATA / 'entities'
LNK = DATA / 'links'

conn = Neo4jConnection().connect()
print('Connected to', conn._uri)

Connected to neo4j+s://692d7258.databases.neo4j.io


## Helpers

- `df_records()` reads a CSV as all-string, drops junk columns, optionally renames.
- `batch_run()` chunks MERGE operations (500 rows per query) so we don't hit the transaction limit on Neo4j AuraDB free tier.

In [2]:
BATCH_SIZE = 500

def df_records(path, rename=None, drop=None):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    if rename:
        df = df.rename(columns=rename)
    if drop:
        df = df.drop(columns=[c for c in drop if c in df.columns])
    return df.to_dict(orient='records')

def batch_run(cypher, records, batch_size=BATCH_SIZE):
    total = 0
    for i in range(0, len(records), batch_size):
        chunk = records[i:i + batch_size]
        conn.run_query(cypher, {'records': chunk})
        total += len(chunk)
    return total

## Optional: reset Layer 1

Flip `RESET = True` and re-run this cell if you want to wipe Layer 1 before reloading. Leaves Layers 2 and 3 untouched.

In [3]:
RESET = False
if RESET:
    for label in ('Person', 'Company', 'Intermediary', 'Address', 'Jurisdiction'):
        conn.run_query(f'MATCH (n:{label}) DETACH DELETE n')
    print('Layer 1 wiped.')
else:
    print('Skipped reset (RESET=False).')

Skipped reset (RESET=False).


## Uniqueness constraints

Neo4j automatically creates a supporting index when we declare a uniqueness constraint. Four labels key off `node_id`; `Jurisdiction` keys off `jurisdiction_id`.

In [4]:
CONSTRAINTS = [
    'CREATE CONSTRAINT person_node_id        IF NOT EXISTS FOR (n:Person)       REQUIRE n.node_id IS UNIQUE',
    'CREATE CONSTRAINT company_node_id       IF NOT EXISTS FOR (n:Company)      REQUIRE n.node_id IS UNIQUE',
    'CREATE CONSTRAINT intermediary_node_id  IF NOT EXISTS FOR (n:Intermediary) REQUIRE n.node_id IS UNIQUE',
    'CREATE CONSTRAINT address_node_id       IF NOT EXISTS FOR (n:Address)      REQUIRE n.node_id IS UNIQUE',
    'CREATE CONSTRAINT jurisdiction_id       IF NOT EXISTS FOR (n:Jurisdiction) REQUIRE n.jurisdiction_id IS UNIQUE',
]
for c in CONSTRAINTS:
    conn.run_query(c)
print(f'Applied {len(CONSTRAINTS)} constraints.')

Applied 5 constraints.


## Load entity nodes

Columns carried in as-is via `SET n += row`. Two housekeeping transformations at load:

- `sourceID` renamed to `source_leak` (more readable property name)
- Redundant `node_type` column dropped (the Neo4j label carries that information)

In [5]:
ICIJ_RENAME = {'sourceID': 'source_leak'}
ICIJ_DROP   = ['node_type']

# Person
records = df_records(ENT / 'persons.csv', rename=ICIJ_RENAME, drop=ICIJ_DROP)
n = batch_run(
    'UNWIND $records AS row MERGE (n:Person {node_id: row.node_id}) SET n += row',
    records,
)
print(f'Person:       {n:>5} nodes')

# Company
records = df_records(ENT / 'companies.csv', rename=ICIJ_RENAME, drop=ICIJ_DROP)
n = batch_run(
    'UNWIND $records AS row MERGE (n:Company {node_id: row.node_id}) SET n += row',
    records,
)
print(f'Company:      {n:>5} nodes')

# Intermediary
records = df_records(ENT / 'intermediaries.csv', rename=ICIJ_RENAME, drop=ICIJ_DROP)
n = batch_run(
    'UNWIND $records AS row MERGE (n:Intermediary {node_id: row.node_id}) SET n += row',
    records,
)
print(f'Intermediary: {n:>5} nodes')

# Address (derived — no sourceID/node_type to drop)
records = df_records(ENT / 'addresses.csv')
n = batch_run(
    'UNWIND $records AS row MERGE (n:Address {node_id: row.node_id}) SET n += row',
    records,
)
print(f'Address:      {n:>5} nodes')

# Jurisdiction (different primary key)
records = df_records(ENT / 'jurisdictions.csv')
n = batch_run(
    'UNWIND $records AS row MERGE (n:Jurisdiction {jurisdiction_id: row.jurisdiction_id}) SET n += row',
    records,
)
print(f'Jurisdiction: {n:>5} nodes')

Person:          19 nodes
Company:        131 nodes
Intermediary:     7 nodes
Address:         28 nodes
Jurisdiction:     3 nodes


## Load relationship edges

Each link CSV already carries `source_id`, `target_id`, `source_type`, `target_type`, `relationship`, `link`, `source_leak`, `status`, `start_date`, `end_date`. We MATCH on `node_id` and MERGE the typed edge.

For `registered_at.csv` the source can be either `Company` or `Intermediary`, so we match generically on `{node_id}` (safe because `node_id` is globally unique in the ICIJ dataset).

In [6]:
# IS_OFFICER_OF: Person -> Company
records = df_records(LNK / 'is_officer_of.csv')
n = batch_run(
    '''
    UNWIND $records AS row
    MATCH (s:Person  {node_id: row.source_id})
    MATCH (t:Company {node_id: row.target_id})
    MERGE (s)-[r:IS_OFFICER_OF]->(t)
    SET r.role        = row.relationship,
        r.source_leak = row.source_leak,
        r.status      = row.status,
        r.start_date  = row.start_date,
        r.end_date    = row.end_date,
        r.link        = row.link
    ''',
    records,
)
print(f'IS_OFFICER_OF:       {n:>5} edges')

# INTERMEDIARY_OF: Intermediary -> Company
records = df_records(LNK / 'intermediary_of.csv')
n = batch_run(
    '''
    UNWIND $records AS row
    MATCH (s:Intermediary {node_id: row.source_id})
    MATCH (t:Company      {node_id: row.target_id})
    MERGE (s)-[r:INTERMEDIARY_OF]->(t)
    SET r.source_leak = row.source_leak,
        r.status      = row.status,
        r.start_date  = row.start_date,
        r.end_date    = row.end_date,
        r.link        = row.link
    ''',
    records,
)
print(f'INTERMEDIARY_OF:     {n:>5} edges')

# REGISTERED_AT: Company | Intermediary -> Address
records = df_records(LNK / 'registered_at.csv')
n = batch_run(
    '''
    UNWIND $records AS row
    MATCH (s              {node_id: row.source_id})
    MATCH (t:Address      {node_id: row.target_id})
    MERGE (s)-[r:REGISTERED_AT]->(t)
    SET r.source_leak = row.source_leak,
        r.link        = row.link
    ''',
    records,
)
print(f'REGISTERED_AT:       {n:>5} edges')

# SHARES_ADDRESS_WITH: Company -> Company (derived red flag)
records = df_records(LNK / 'shares_address_with.csv')
n = batch_run(
    '''
    UNWIND $records AS row
    MATCH (s:Company {node_id: row.source_id})
    MATCH (t:Company {node_id: row.target_id})
    MERGE (s)-[r:SHARES_ADDRESS_WITH]->(t)
    SET r.address_node_id = row.link,
        r.source_leak     = row.source_leak
    ''',
    records,
)
print(f'SHARES_ADDRESS_WITH: {n:>5} edges')

# SIMILAR_TO: Company -> Company (ICIJ-flagged)
records = df_records(LNK / 'similar.csv')
n = batch_run(
    '''
    UNWIND $records AS row
    MATCH (s:Company {node_id: row.source_id})
    MATCH (t:Company {node_id: row.target_id})
    MERGE (s)-[r:SIMILAR_TO]->(t)
    SET r.source_leak = row.source_leak,
        r.link        = row.link
    ''',
    records,
)
print(f'SIMILAR_TO:          {n:>5} edges')

IS_OFFICER_OF:         158 edges
INTERMEDIARY_OF:        98 edges
REGISTERED_AT:         138 edges
SHARES_ADDRESS_WITH:    23 edges
SIMILAR_TO:              2 edges


## Derive `INCORPORATED_IN`

Links each `Company` to the `Jurisdiction` whose ID matches the company's `jurisdiction` property (ISO alpha-3 code). Cheap to derive and saves us authoring another CSV.

In [7]:
result = conn.run_query('''
    MATCH (c:Company), (j:Jurisdiction)
    WHERE c.jurisdiction = j.jurisdiction_id AND c.jurisdiction <> ''
    MERGE (c)-[:INCORPORATED_IN]->(j)
    RETURN count(*) AS edges
''')
print(f'INCORPORATED_IN:     {result[0]["edges"]:>5} edges')

INCORPORATED_IN:       131 edges


## Verification

In [8]:
print('Nodes by label:')
for row in conn.run_query('MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY n DESC'):
    print(f'  {row["label"]:<16} {row["n"]:>5}')

print('\nRelationships by type:')
for row in conn.run_query('MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY n DESC'):
    print(f'  {row["rel"]:<22} {row["n"]:>5}')

print('\nSample PEPs and their shell counts (top 10):')
for row in conn.run_query('''
    MATCH (p:Person)-[:IS_OFFICER_OF]->(c:Company)
    WITH p, count(c) AS shells
    WHERE shells >= 2
    RETURN p.name AS name, p.countries AS country, shells
    ORDER BY shells DESC LIMIT 10
'''):
    print(f'  {row["name"]:<40} {(row["country"] or ""):<20} {row["shells"]:>3} shells')

Nodes by label:
  Company            131
  Address             28
  Person              19
  Intermediary         7
  Jurisdiction         3

Relationships by type:
  IS_OFFICER_OF            147
  REGISTERED_AT            138
  INCORPORATED_IN          131
  INTERMEDIARY_OF           98
  SHARES_ADDRESS_WITH       23

Sample PEPs and their shell counts (top 10):
  MINERVA NOMINEES LIMITED                 Jersey               118 shells
  MINERVA SERVICES LIMITED                 Jersey                 6 shells
  MINERVA NOMINEES LIMITED                 Jersey                 6 shells
  Hussain Nawaz Sharif                                            2 shells


In [9]:
conn.close()
print('Done.')

Done.
